# Breakout 4B · Informality: matching customs and tax

Workshop **Data Driven Domestic Revenue Mobilization**, Bangkok, October 2026. Co-run with the tax
facilitator: every result says which side acts.

Match customs records to the tax registry with and without the tax ID, choose how strict the match is
(the dial), sort importers into four buckets, and find the two networks that borrow names.

Press ▶ on each cell from top to bottom. Change only the sliders and drop-down menus. Everything here
is synthetic: every trader and name is generated.

In [ ]:
#@title 0 · Where the data comes from
#@markdown Leave these as they are. Your analysts can point DATA_URL at their own extract (section 6).
DATA_URL = "https://raw.githubusercontent.com/FrancoisChastel/CustomsWorkshopNotebooks/data-v2/data"  #@param {type:"string"}
INSTALL = True  #@param {type:"boolean"}

In [ ]:
#@title 0 · Setup (run me first)
import sys, subprocess, warnings
warnings.filterwarnings("ignore")
if INSTALL and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/FrancoisChastel/CustomsWorkshopNotebooks@data-v2"], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from workshop_methods import notebook as nb
from workshop_methods.data import load_json, load_tables
from workshop_methods import border, informality, signals
nb.use_style()
pd.set_option("display.max_colwidth", 40)
T = load_tables(DATA_URL)
print(f"Loaded {len(T)} tables, {len(T['declarations']):,} declaration lines. Ready.")

In [ ]:
#@title 0 · Prepare the tables
d, traders, registry, returns = T["declarations"], T["traders"], T["tax_registry"], T["vat_returns"]
months = sorted(d["month"].unique())
imp = d[(d["flow"] == "import") & (d["procedure"] == "IM4") & d["month"].isin(months[-12:])]
value_12m = imp.groupby("trader_ref")["customs_value_lcu"].sum()
importers = traders[traders["trader_ref"].isin(value_12m.index)].copy()
importers["import_value_12m"] = importers["trader_ref"].map(value_12m)
print(f"{len(importers)} importers in the last 12 months; the tax registry holds {len(registry)} taxpayers.")

## 1 · Exact match: join on the tax ID

In [ ]:
#@title 1 · Join on the tax ID and explain every miss
nb.steps("How many importers can we find in the tax registry by their tax ID?",
         "Join each importer's tax ID, as customs holds it, to the registry.",
         "Give the reason for every miss: no ID, an ID one character off, or an ID the registry does not hold.")
exact = informality.exact_match(importers, registry)
counts = exact["exact_status"].value_counts().rename("importers").to_frame()
counts["import_value_12m"] = exact.groupby("exact_status")["import_value_12m"].sum()
nb.show(counts, formats={"import_value_12m": "{:,.0f}"})
off = exact[exact["exact_status"] == "TIN one character off"]
nb.finding(f"{counts.at['matched', 'importers']} of {len(exact)} importers match on the tax ID; "
           f"{counts['importers'].drop('matched').sum()} miss, {len(off)} of them because customs holds the ID one character off. "
           "Customs and tax act together: validate IDs at registration with a check digit.")

## 2 · Probable match and the dial

In [ ]:
#@title 2 · Match on name, address, phone and director, and set the dial
PRESET = "balanced"  #@param ["strict", "balanced", "loose", "use the slider"]
THRESHOLD = 85  #@param {type:"slider", min:50, max:100, step:5}
threshold = {"strict": 95, "balanced": 85, "loose": 70}.get(PRESET, THRESHOLD)
nb.steps("When the ID fails, can names, addresses, phones and directors find the taxpayer?",
         "Clean the names (lower case, legal suffixes removed); compare only pairs with the same name start and region.",
         "Score name similarity (0 to 100, weighted 0.7) plus 10 points each for the same address, phone and director.",
         f"Accept pairs scoring {threshold} or more.")
scores = informality.probable_scores(importers, registry)
dial = pd.DataFrame(load_json(DATA_URL, "dial_4b")["rows"]).set_index("threshold")
fig, ax = nb.chart(f"At {threshold}: {dial.at[threshold, 'found']} found, {dial.at[threshold, 'wrong']} wrong, "
                   f"{dial.at[threshold, 'missed']} missed")
ax.plot(dial.index, dial["found"], color=nb.COLORS["accent"], label="found (right taxpayer)")
ax.plot(dial.index, dial["wrong"] * 10, color=nb.COLORS["amber"], label="wrong (x10 to be visible)")
ax.axvline(threshold, color=nb.COLORS["ink"], linestyle=":")
ax.set_xlabel("threshold"); ax.set_ylabel("importers"); ax.legend(); plt.show()
trap = off.merge(scores, on="trader_ref", how="left")
caught = trap[trap["score"] >= threshold]
nb.finding(f"Strict (95) is for automatic data exchange: {dial.at[95, 'wrong']} wrong, {dial.at[95, 'missed']} missed. "
           f"Balanced (85) is for a case list a person checks: {dial.at[85, 'wrong']} wrong, {dial.at[85, 'missed']} missed. "
           f"Loose (70) is for exploring: {dial.at[70, 'wrong']} wrong. At {threshold}, {len(caught)} of the {len(trap)} importers "
           "with an ID one character off are found by their name, phone and director.")

## 3 · Four buckets

In [ ]:
#@title 3 · Unregistered, not filing, under-reporting, consistent
MARKUP = 1.25  #@param {type:"number"}
VAT_THRESHOLD = 1_500_000  #@param {type:"integer"}
nb.steps("Which large importers does the VAT office need to act on, and how?",
         f"Take importers above the VAT threshold ({VAT_THRESHOLD:,} LCU of imports in 12 months) and their matched tax ID.",
         "Unregistered: no match, or not registered for VAT. Not filing: returns missing in the last six months.",
         f"Under-reporting: sales declared below imports x {MARKUP} (the sector mark-up). Consistent: the rest.")
matched = exact.set_index("trader_ref")["trader_id"].where(exact.set_index("trader_ref")["exact_status"] == "matched")
probable = scores.set_index("trader_ref")["matched_tin"].where(scores.set_index("trader_ref")["score"] >= 85)
large = importers[importers["import_value_12m"] > VAT_THRESHOLD].copy()
large["matched_tin"] = large["trader_ref"].map(matched).fillna(large["trader_ref"].map(probable)).fillna("")
sorted_ = informality.buckets(large, registry, returns, months, markup=MARKUP).merge(large[["trader_ref", "import_value_12m"]], on="trader_ref")
per = sorted_.groupby("bucket").agg(importers=("trader_ref", "size"), import_value_12m=("import_value_12m", "sum"))
per = per.reindex(["unregistered", "not filing", "under-reporting", "consistent"]).fillna(0)
lead = {"unregistered": ("tax", "register for VAT; customs shares the import list"),
        "not filing": ("tax", "filing reminder, then an estimated assessment"),
        "under-reporting": ("tax, with customs data", "desk audit of sales against imports"),
        "consistent": ("none", "keep in the risk model")}
per["lead agency"] = [lead[b][0] for b in per.index]
per["action"] = [lead[b][1] for b in per.index]
fig, ax = nb.chart("Imports by bucket, last 12 months")
ax.bar(per.index, per["import_value_12m"] / 1e6, color=[nb.COLORS["amber"], nb.COLORS["amber"], nb.COLORS["accent"], nb.COLORS["muted"]])
ax.set_ylabel("LCU million"); plt.show()
nb.show(per, formats={"importers": "{:.0f}", "import_value_12m": "{:,.0f}"})
nb.finding(f"Of {len(large)} importers above the threshold, {per.at['unregistered', 'importers']:.0f} are unregistered, "
           f"{per.at['not filing', 'importers']:.0f} not filing and {per.at['under-reporting', 'importers']:.0f} under-reporting "
           f"(LCU {per.loc[['unregistered', 'not filing', 'under-reporting'], 'import_value_12m'].sum():,.0f} of imports). "
           "Tax leads on all three; customs supplies the import data.")

## 4 · Borrowed names and patterns

In [ ]:
#@title 4 · Who clears for many consignees at one address?
nb.steps("Does anyone lend its tax ID to others?",
         "Draw traders, the consignees they clear for, their addresses and brokers as a graph.",
         "Count consignees per tax ID and address; find groups of nodes tied together (communities).")
import networkx as nx
net = load_json(DATA_URL, "network")
keep = {"trader", "consignee", "address", "broker"}
G = border.build_graph({"nodes": [n for n in net["nodes"] if n["kind"] in keep],
                        "edges": [e for e in net["edges"] if e["kind"] != "buys from"]})
communities = nx.community.greedy_modularity_communities(G)
clusters = border.shared_address_clusters(T["consignees"])
consignees = T["consignees"]
def usual_broker(trader, address):
    ids = consignees[(consignees["trader_ref"] == trader) & (consignees["address_id"] == address)]["consignee_id"]
    brokers = d.loc[d["consignee_id"].isin(ids), "broker_id"].mode()
    return brokers.iloc[0] if len(brokers) else ""
clusters["broker"] = [usual_broker(t, a) for t, a in zip(clusters["trader_ref"], clusters["address_id"])]
clusters["tax_id"] = clusters["trader_ref"].map(traders.set_index("trader_ref")["trader_id"])
nb.show(clusters[["tax_id", "address_id", "consignees", "broker"]].head(6).reset_index(drop=True))
top2 = clusters.head(2)
nb.finding(f"Tax IDs {top2['tax_id'].iloc[0]} and {top2['tax_id'].iloc[1]} each clear for {top2['consignees'].iloc[0]} and "
           f"{top2['consignees'].iloc[1]} consignees sharing one address, through one broker each ({top2['broker'].iloc[0]}, "
           f"{top2['broker'].iloc[1]}); the graph splits into {len(communities)} groups. Customs and tax visit together.")

In [ ]:
#@title 4 · Pattern: parcels split under the duty-free limit
split = border.parcel_splitting(T["parcels"])
repeat = split[split["months_flagged"] >= 3]
nb.show(split.head(6), formats={"max_value_lcu": "{:,.0f}"})
nb.finding(f"{len(repeat)} receivers get six parcels or more a month just under the limit; neither duty nor VAT is paid. "
           "Customs acts: aggregate parcels by receiver before applying the limit.")

In [ ]:
#@title 4 · Pattern: the land border, crossing by crossing
lbm = T["land_border_mirror"].assign(ratio=lambda f: f["home_imports_lcu"] / f["neighbour_exports_lcu"])
fig, ax = nb.chart("Home imports against the neighbour's exports, by crossing")
for office, part in lbm.groupby("crossing_office"):
    part = part.sort_values("month")
    ax.plot(part["month"], part["ratio"], label=f"{office} ({part['neighbour'].iloc[0]})")
ax.axhline(1.0, color=nb.COLORS["muted"], linestyle=":"); ax.set_xticks(sorted(lbm["month"].unique())[::4]); ax.legend(); plt.show()
trend = {o: np.polyfit(range(len(p)), p.sort_values("month")["ratio"], 1)[0] for o, p in lbm.groupby("crossing_office")}
worst = min(trend, key=trend.get)
part = lbm[lbm["crossing_office"] == worst].sort_values("month")
nb.finding(f"At {worst} home imports fall from {part['ratio'].iloc[0]:.0%} to {part['ratio'].iloc[-1]:.0%} of what "
           f"{part['neighbour'].iloc[0]} records as exported through it; the other crossing stays near parity. "
           "Customs acts with the neighbour: joint checks at the crossing.")

In [ ]:
#@title 4 · Pattern: a sector that sells less than it imports
sales = returns[returns["period"].isin(months[-12:])].groupby("tin")["sales_declared"].sum()
frame = importers.assign(sales=importers["trader_id"].map(sales)).dropna(subset=["sales"])
share = (frame["sales"] < frame["import_value_12m"]).groupby(frame["sector"]).mean().sort_values(ascending=False)
fig, ax = nb.chart(f"{share.index[0]}: {share.iloc[0]:.0%} of importers declare sales below their imports")
ax.barh(share.index[::-1], share.values[::-1] * 100, color=[nb.COLORS["amber"] if s == share.index[0] else nb.COLORS["accent"] for s in share.index[::-1]])
ax.set_xlabel("importers selling less than they import, %"); plt.show()
nb.finding(f"In {share.index[0]}, {share.iloc[0]:.0%} of importers declare sales below their imports, against "
           f"{share.drop(share.index[0]).median():.0%} in the median sector: sales in cash. Tax acts, with customs import data.")

In [ ]:
#@title 4 · Pattern: origin washed through one seller
check = signals.origin_check(imp, T["tariff"], d[d["flow"] == "import"])
doubtful = imp[check["doubtful_claim"]]
sellers = doubtful.groupby("exporter_id").agg(importers=("trader_ref", "nunique"), lines=("decl_id", "size")).sort_values("importers", ascending=False)
nb.show(sellers.head(5))
nb.finding(f"Sellers {sellers.index[0]} and {sellers.index[1]} are behind most claims from origins that do not make the goods "
           f"({sellers['importers'].iloc[0]} and {sellers['importers'].iloc[1]} importers). Customs acts: verify the certificates.")

In [ ]:
#@title 4 · Pattern: one tax ID, many names
per_id = T["consignees"].groupby("trader_ref").agg(consignees=("consignee_id", "size"), addresses=("address_id", "nunique"))
per_id["consignees_per_address"] = (per_id["consignees"] / per_id["addresses"]).round(1)
per_id = per_id.sort_values("consignees_per_address", ascending=False)
per_id.index = per_id.index.map(traders.set_index("trader_ref")["trader_id"])
nb.show(per_id.head(5))
nb.finding(f"Tax IDs {per_id.index[0]} and {per_id.index[1]} clear for {per_id['consignees_per_address'].iloc[0]:.0f} and "
           f"{per_id['consignees_per_address'].iloc[1]:.0f} consignees per address; a group clearing for its own subsidiaries "
           "has one address each. Both sides act: a joint visit.")

## 5 · Exercise: 25 importers

Roles: the **analyst** reads the table, the **officer** sorts each importer into a bucket and chooses the action
and the lead agency, the **committee** marks the actions that need a new data-sharing step. Then name the
two networks. Nothing about the answer is visible until the reveal.

In [ ]:
#@title 5 · The 25 importers
ws = pd.DataFrame(load_json(DATA_URL, "worksheet_4b")["rows"]).set_index("ref")
nb.show(ws[["importer", "sector", "consignees", "import_value_12m", "registry_match", "returns_filed_12m", "sales_declared_12m"]],
        formats={"import_value_12m": "{:,.0f}", "sales_declared_12m": "{:,.0f}"}, max_rows=25)
nb.finding(f"{(~ws['registry_match']).sum()} of the 25 have no exact registry match and {(ws['returns_filed_12m'] < 12).sum()} filed "
           "fewer than twelve returns: start from those, then compare sales with imports.")

In [ ]:
#@title 5 · Your buckets, actions and leads
import ipywidgets as widgets
from IPython.display import display
BUCKETS, ACTIONS = ['', 'unregistered', 'not filing', 'under-reporting', 'consistent'], {'': ['', False], 'register for VAT': ['tax', False], 'filing reminder, then estimated assessment': ['tax', False], 'desk audit: sales against imports': ['tax', True], 'valuation query or post-clearance audit': ['customs', False], 'joint visit': ['customs and tax', True], 'no action: keep in the risk model': ['', False]}
rows = [widgets.HBox([widgets.Label(i, layout=widgets.Layout(width="90px")),
                      widgets.Dropdown(options=BUCKETS, layout=widgets.Layout(width="140px")),
                      widgets.Dropdown(options=list(ACTIONS), layout=widgets.Layout(width="280px")),
                      widgets.Dropdown(options=["", "customs", "tax", "customs and tax"], layout=widgets.Layout(width="140px")),
                      widgets.Checkbox(description="new data-sharing step", indent=False)]) for i in ws.index]
networks = widgets.Text(description="Networks", placeholder="the two lead tax IDs, comma separated")
lock = widgets.Button(description="Lock my answers", button_style="primary")
note = widgets.Output()
LOCKED = {"buckets": {}, "actions": {}, "networks": []}
def on_lock(_):
    LOCKED["buckets"] = {r.children[0].value: r.children[1].value for r in rows}
    LOCKED["actions"] = {r.children[0].value: r.children[2].value for r in rows}
    LOCKED["networks"] = [t.strip() for t in networks.value.split(",") if t.strip()]
    with note:
        note.clear_output()
        print(f"Locked: {sum(1 for v in LOCKED['buckets'].values() if v)} of 25 importers bucketed; networks: "
              f"{', '.join(LOCKED['networks']) or 'none'}.")
lock.on_click(on_lock)
display(widgets.VBox(rows + [networks, lock, note]))
#@markdown If the menus do not show, type "I01: unregistered, I02: consistent, ..." and the two networks here instead.
MY_BUCKETS = ""  #@param {type:"string"}
MY_NETWORKS = ""  #@param {type:"string"}
if MY_BUCKETS or MY_NETWORKS:
    LOCKED["buckets"] = dict(p.split(":", 1) for p in MY_BUCKETS.replace(" ", "").split(",") if ":" in p)
    LOCKED["networks"] = [t.strip() for t in MY_NETWORKS.split(",") if t.strip()]
    print(f"Typed in: {len(LOCKED['buckets'])} buckets; networks {', '.join(LOCKED['networks'])}.")

## Reveal (facilitator)

The facilitator types the code. Without it, this cell prints nothing.

In [ ]:
#@title 5 · Reveal
REVEAL_CODE = ""  #@param {type:"string"}
SEALED = {"salt": "VfmgRimI8zeWT6UlEZDTTw==", "data": "meovQAoaaj/clgyvnG4arJ6MQKs6V9IM4rdM0bul6k50qa5Z+FrsHIMdj1L7pSVYqqIDwlSkGnvl0vllkQTS+2OFXoaxsZid+rjzM8eWFSqE0OHTgK7cbtKx+xw4RA74ZlZNxU4pWor/xzPKYC+DoyhG2+ZJCX4JVAQwd15pEFbEQIJ2K5bezhGyW+XXWX+mLSNOvWt3jt0xPZgAOyigdtXnQpGzNrzh8GfjN+6cNRres+YAjKCq4f/G0RlKWskIXRuiLjphVLz6nHGW5wrqFNQEmvealnS3B3VjBA8wqc7GqcCscy1q85SxiV+wi0kvdFHg2yXyHisOQEtrmqeQrR6r/DiJ7qtCq8vCBoFqf1ooAjIEim+gH9ZGNsS2BVFar+g6F5dbcq5qot36sP7CrorKifp2YldbaHokm7mzZN6Zp9ndLLqP+u5vqZ6+5AtQB9KkudVVPouGEE9icBLCR+eyx5ysj4t2+UWHNn7KXcxSsEHBrfHm82rNNUomL73P8648QANt2U4dqRSUXBFQpUtiLPkXSFhCh0kfuiYk46h+UgUIiYvdQWprcoVlNFFIuir47QNePdOk7yGh1npLKc0cDS8qC/cP2pDU2aHAV7nGMK3DlG62gCPUk+KlhzvZnJDrCkKSzz21YiOOhXmzZSt0JLAXWtu6YJdXDWJ2bznCb7cE36KiAqaHN2g++Hk7BrWlbIO27AEmPcFR4YUSa5RDnSFPEFV1pRpfdvpW7FeytQzk41Ydlb3qiiZIchy+L6X2j7h48lB5R27eK7v2EhbAnD4ZqGbyVHf7s9SWq3Dp3Vu8V/K0vl+iN5M6lC9Vw7fcQ9MloOCOt5UxP/Xaq8tCc+SBOzyWZisTYO+eStdGBaE8dd190rh0qLJO1g+MIZTVlCEtnQ8s8lPlGbiqr8zFqvI8eFDnszDdqjbXabStXqjE4KphkXCqZFmTOAdrvbbZZfwE6OXy+UObF47Kf6fya/EViqf7TGkp", "mac": "328f32b4d7859b6a9927aa4eb805da95d8db3515ea2950db6fd60340d736621b"}
answer = nb.unseal(REVEAL_CODE, SEALED)
if answer:
    key = pd.DataFrame(answer["rows"]).set_index("ref")
    key["your bucket"] = key.index.map(lambda i: LOCKED["buckets"].get(i, "").replace("-", " ").replace("_", " "))
    key["right"] = key["your bucket"] == key["bucket"].str.replace("_", " ").str.replace("-", " ")
    key["import_value_12m"] = ws["import_value_12m"].reindex(key.index)
    nb.show(key[["importer", "bucket", "your bucket", "right", "why"]], max_rows=25, highlight=~key["right"])
    per = key.groupby("bucket")["import_value_12m"].agg(["size", "sum"])
    nets = pd.DataFrame(answer["networks"])
    found = len(set(LOCKED["networks"]) & set(nets["lead_tin"]))
    chosen = [a for a in LOCKED["actions"].values() if a]
    today = sum(1 for a in chosen if not ACTIONS[a][1])
    nb.finding(f"You bucketed {int(key['right'].sum())} of 25 importers right and found {found} of the 2 networks "
               f"({'; '.join(f'{r.lead_tin}: {r.consignees} consignees at {r.shared_address}, broker {r.shared_broker}' for r in nets.itertuples())}). "
               f"{today} of your {len(chosen)} actions are possible today without a new agreement.")
    nb.show(per.rename(columns={"size": "importers", "sum": "import_value_12m"}), formats={"import_value_12m": "{:,.0f}"})

## 6 · For your analysts

**Deterministic and probabilistic linkage.** An exact join on the tax ID is deterministic: it is right when
it matches and silent when it does not. Probable matching scores every candidate pair and accepts those above
a threshold; it finds what the ID misses and sometimes accepts the wrong taxpayer.

**Blocking.** Comparing every customs record with every registry record is slow; compare only pairs that share
something cheap to check (here the first three letters of the cleaned name and the region).

**Choosing the threshold.** Hand-check a random sample of pairs at a few thresholds and plot found against
wrong (precision and recall). Strict thresholds suit automatic exchange; balanced ones suit case lists a person
checks. Splink is the open-source tool most administrations use in production.

**Governance checklist.** A shared identifier and its validation; outcomes recorded on both sides; a named
owner for the match; the legal basis for the exchange; a fairness check (who is matched wrongly most often);
privacy (minimum fields, retention, access log).

In [ ]:
#@title 6 · Matching in a few lines
exact = informality.exact_match(importers, registry)                      # deterministic: join on the tax ID
scores = informality.probable_scores(importers, registry)                 # probabilistic: name, address, phone, director
accepted = scores[scores["score"] >= 85]                                  # the dial
print(exact["exact_status"].value_counts().to_string(), f"\nprobable matches at 85: {len(accepted)}")